# [1] Setup and Imports
Import all necessary libraries, set random seeds, and configure visualization styles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Set styles
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')
pd.set_option('display.max_columns', None)

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)


# [2] Load and Inspect Data
Load the synthetic dataset generated earlier and inspect its structure, data types, and check for missing values.

In [ ]:
# Load Data
try:
    df = pd.read_csv('../data/sample_transactions.csv')
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Dataset not found. Please run data/generate_data.py first.")

# Inspect
print("Shape:", df.shape)
display(df.head())

# Missing values
missing = df.isnull().sum()
print("Missing values:\n", missing[missing > 0])


# [3] Univariate Analysis
Examine the distribution of individual numeric and categorical variables.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Quantity
sns.histplot(df['quantity'], bins=5, ax=axes[0, 0], kde=False, color='cyan')
axes[0, 0].set_title('Quantity Distribution')

# Unit Price
sns.histplot(df['unit_price'], bins=30, ax=axes[0, 1], kde=True, color='magenta')
axes[0, 1].set_title('Unit Price Distribution')

# Category count
sns.countplot(y='product_category', data=df, ax=axes[1, 0], palette='plasma', order=df['product_category'].value_counts().index)
axes[1, 0].set_title('Category Count')

# Payment Method
sns.countplot(x='payment_method', data=df, ax=axes[1, 1], palette='viridis')
axes[1, 1].set_title('Payment Method Count')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


# [4] Bivariate Analysis
Analyze relationships between two variables, such as revenue by category and customer segment behaviors.

In [ ]:
# Revenue by category
rev_by_cat = df.groupby('product_category')['total_price'].sum().reset_index()
fig = px.bar(rev_by_cat, x='product_category', y='total_price', color='total_price', 
             title="Total Revenue by Category", template="plotly_dark", color_continuous_scale='plasma')
fig.show()

# Order value by Segment
fig2 = px.box(df, x='customer_segment', y='total_price', color='customer_segment',
              title="Order Value Distribution by Customer Segment", template="plotly_dark")
fig2.show()


# [5] Time Series Analysis
Analyze purchasing trends over time, including monthly aggregates and weekday vs weekend patterns.

In [ ]:
monthly_revenue = df.set_index('transaction_date').resample('ME')['total_price'].sum().reset_index()
fig = px.line(monthly_revenue, x='transaction_date', y='total_price', markers=True,
              title="Monthly Revenue Trend", template="plotly_dark")
fig.show()

# Day of week
df['day_of_week'] = df['transaction_date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = df['day_of_week'].value_counts().reindex(day_order).reset_index()
fig2 = px.bar(dow_counts, x='day_of_week', y='count', title="Transactions by Day of Week", template="plotly_dark", color='count')
fig2.show()


# [6] Customer Segmentation Analysis
Evaluate customer behavior using RFM (Recency, Frequency, Monetary) metrics.

In [ ]:
import datetime as dt
max_date = df['transaction_date'].max()

rfm = df.groupby('customer_id').agg({
    'transaction_date': lambda x: (max_date - x.max()).days, # Recency
    'transaction_id': 'nunique',                             # Frequency
    'total_price': 'sum'                                     # Monetary
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

fig = px.scatter_3d(rfm, x='Recency', y='Frequency', z='Monetary', 
                    color='Monetary', opacity=0.7, title="RFM 3D Scatter", template="plotly_dark")
fig.show()


# [7] Product Analysis
Identify top-performing products and understand category-level distributions.

In [ ]:
top_products = df.groupby('product_name')['total_price'].sum().nlargest(20).reset_index()
fig = px.bar(top_products, x='total_price', y='product_name', orientation='h', 
             title="Top 20 Products by Revenue", template="plotly_dark", color='total_price')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# Treemap
cat_treemap = df.groupby(['product_category', 'product_name'])['total_price'].sum().reset_index()
fig2 = px.treemap(cat_treemap, path=['product_category', 'product_name'], values='total_price',
                  title="Revenue Treemap: Category > Product", template="plotly_dark")
fig2.show()


# [8] Co-Purchase / Basket Analysis
Investigate which products are frequently bought together by building a co-occurrence matrix.

In [ ]:
from itertools import combinations
from collections import Counter

# Get transactions with > 1 item
basket = df.groupby('transaction_id')['product_name'].apply(list).reset_index()
basket['num_items'] = basket['product_name'].apply(len)
multi_item_baskets = basket[basket['num_items'] > 1]

# Count pairs
pairs = Counter()
for items in multi_item_baskets['product_name']:
    # Sort to ensure (A, B) is same as (B, A)
    items.sort()
    for pair in combinations(items, 2):
        pairs[pair] += 1

top_pairs = pairs.most_common(15)
pair_names = [f"{p[0][0]} & {p[0][1]}" for p in top_pairs]
pair_counts = [p[1] for p in top_pairs]

fig = px.bar(x=pair_counts, y=pair_names, orientation='h', title="Top 15 Most Frequent Product Pairs", template="plotly_dark")
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()


# [9] Geographic Analysis
Explore transaction volumes and revenue distributions across different cities.

In [ ]:
city_rev = df.groupby('city')['total_price'].sum().reset_index().sort_values('total_price', ascending=False)
fig = px.bar(city_rev, x='city', y='total_price', title="Revenue by City", template="plotly_dark", color='total_price')
fig.show()


# [10] Key Insights Summary
- **Revenue Concentration:** A few top products drive a significant portion of revenue.
- **Seasonality:** Specific months show distinct peaks.
- **Customer Segmentation:** High-value customers exist and can be targeted differently.
- **Co-purchases:** Clear associations exist between specific items, validating the need for Association Rule mining and Collaborative Filtering.

**Cross-sell Opportunities:**
1. Bundle high-affinity pairs from the Co-Purchase analysis.
2. Target "Budget" segment with highly rated but lower-cost items.
3. Use geographic preferences to personalize landing pages.
